# Transformers in Generative AI

Every modern large language model — GPT, LLaMA, Claude, Gemini — and many image generators — Stable Diffusion, DALL·E, Midjourney — are built on the **Transformer** architecture. Before Transformers, sequence generation relied on recurrent networks that processed tokens one at a time, struggled with long-range dependencies, and resisted parallelization on GPUs. The Transformer replaced recurrence with **self-attention**, enabling models to relate every token to every other token in a single parallel pass and to scale to billions of parameters.

This notebook focuses on Transformers **as they are used in generative AI**: decoder-only language models that predict the next token, encoder-decoder models for conditional generation, and the attention mechanisms that power diffusion-based image synthesis. We walk through each component — embeddings, multi-head attention, positional encoding, feed-forward layers, and residual connections — and implement the core operations in NumPy so you can see exactly how data flows through a generative Transformer block.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(7)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. Transformers in Generative AI

Generative AI requires models that can **produce** structured outputs — sequences of tokens, pixels, or latent vectors — conditioned on context or random noise. Transformers excel at this for three reasons:

**Parallel training.** Unlike RNNs, which update a hidden state sequentially, self-attention computes relationships between all positions simultaneously. A GPU can process an entire sequence of 2,048 tokens in one forward pass, dramatically reducing training time.

**Long-range dependencies.** When generating the word *"it"* in *"The animal didn't cross the street because it was tired,"* the model must connect *"it"* to *"animal"* across many intervening tokens. Attention creates direct paths between any two positions, regardless of distance.

**Scalability.** Stacking identical Transformer blocks and increasing width (embedding dimension) and depth (number of layers) yields predictable improvements. This scaling law drove the jump from GPT-2 (1.5B parameters) to GPT-4 and beyond.

Transformers power generative AI across modalities:

| Modality | Generative use | Transformer role |
|----------|---------------|------------------|
| **Text** | Autoregressive next-token prediction | Decoder-only stacks (GPT, LLaMA) |
| **Text (conditional)** | Translation, summarization | Encoder-decoder (T5, BART) |
| **Images** | Latent diffusion, text-to-image | U-Net with cross-attention to text encoder |
| **Multimodal** | Image captioning, visual QA | Vision encoder + language decoder |

The unifying idea: **attention lets the model decide which parts of the input matter most when producing each output element.** In text generation, each new token attends to all previous tokens. In image diffusion, each spatial region attends to the text prompt embedding. The same mechanism, different data shapes.

---

## 2. Architecture of Transformers in Gen AI

The original Transformer (Vaswani et al., 2017) has two stacks: an **encoder** that builds rich representations of input, and a **decoder** that generates output while attending to the encoder. Modern generative systems use one stack or the other, depending on the task.

### 2.1 Decoder-Only (Autoregressive Generation)

**Decoder-only** models — GPT, LLaMA, Mistral — are the dominant architecture for text generation. Each layer applies **causal (masked) self-attention**: token $i$ may attend only to tokens at positions $\leq i$, never to future tokens. The model is trained to predict the next token given all previous tokens:

$$P(x_1, x_2, \ldots, x_n) = \prod_{t=1}^{n} P(x_t \mid x_1, \ldots, x_{t-1})$$

At inference time, the model generates one token at a time, appends it to the context, and repeats until a stop condition. This is **autoregressive generation** — the core loop behind ChatGPT-style assistants.

```
Prompt: "The capital of France is"
         ↓
  [Decoder-only Transformer]
         ↓
  Next-token distribution → sample " Paris"
         ↓
  Append, repeat → " Paris." → " Paris. It" → ...
```

### 2.2 Encoder-Decoder (Conditional Generation)

**Encoder-decoder** models — T5, BART, the original Transformer — suit tasks where input and output differ in length or structure: translation, summarization, question answering. The encoder processes the full input bidirectionally. The decoder generates output tokens using **self-attention** (causal) plus **cross-attention** to encoder states.

### 2.3 Which Architecture for Generation?

| Architecture | Attention pattern | Best for |
|-------------|-------------------|----------|
| **Decoder-only** | Causal self-attention | Open-ended text, code, dialogue |
| **Encoder-decoder** | Bidirectional encoder + causal decoder + cross-attention | Translation, summarization, structured I/O |
| **Encoder-only** | Bidirectional (not generative) | Understanding tasks — classification, not generation |

For pure generation — completing text, writing stories, generating code — **decoder-only** models dominate because a single stack trained on next-token prediction learns a general-purpose language model that can be prompted for almost any task.

---

## 3. Input Embeddings in Transformers

Transformers operate on **continuous vectors**, not raw text. The first step converts discrete token IDs into dense **embeddings** — learned lookup tables that map each vocabulary entry to a vector of dimension $d_{\text{model}}$ (typically 768, 4096, or larger in modern LLMs).

Given a sequence of token IDs $[t_1, t_2, \ldots, t_n]$, the embedding layer produces a matrix $\mathbf{X} \in \mathbb{R}^{n \times d_{\text{model}}}$ where each row is the embedding of one token.

**Why embeddings matter for generation:**

- They capture **semantic similarity** — related words ("king", "queen") end up near each other in vector space after training.
- They provide a **shared representation** that all subsequent layers (attention, FFN) operate on.
- In subword tokenizers (BPE, SentencePiece), rare words decompose into frequent subword embeddings, enabling open-vocabulary generation.

In GPT-style models, the embedding matrix is often **tied** to the output projection layer — the same weights map token IDs to vectors on input and map final hidden states back to vocabulary logits on output. This reduces parameters and can improve training stability.

After embedding lookup, **positional information** is added (Section 5) before the first Transformer block. The combined representation flows through stacked layers of attention and feed-forward networks.

---

## 4. Multi-Head Attention

Attention is the mechanism that lets each token gather information from other tokens. In **self-attention**, queries, keys, and values all come from the same sequence. The operation is **scaled dot-product attention**:

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right) \mathbf{V}$$

- **Query (Q)** — what information does this position need?
- **Key (K)** — what does each position offer for matching?
- **Value (V)** — what content should be passed forward if a match is found?

In self-attention, $\mathbf{Q}$, $\mathbf{K}$, and $\mathbf{V}$ are linear projections of the same input: $\mathbf{Q} = \mathbf{X}\mathbf{W}^Q$, $\mathbf{K} = \mathbf{X}\mathbf{W}^K$, $\mathbf{V} = \mathbf{X}\mathbf{W}^V$.

**Why scale by $\sqrt{d_k}$?** Dot products grow in magnitude as dimension $d_k$ increases, pushing softmax into near one-hot regions with vanishing gradients. Dividing by $\sqrt{d_k}$ keeps score variance stable.

**Multi-head attention** runs $h$ parallel attention operations with separate projection matrices, then concatenates and projects the result:

$$\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}_i^Q, \mathbf{K}\mathbf{W}_i^K, \mathbf{V}\mathbf{W}_i^V)$$

$$\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \mathbf{W}^O$$

Different heads can specialize — one may track syntactic dependencies, another positional proximity, another semantic coreference. GPT-3 uses 96 heads; LLaMA 2 70B uses 64.

**Causal masking in generative models:** Decoder-only models apply a mask so position $i$ cannot attend to positions $j > i$. Future tokens are set to $-\infty$ before softmax, ensuring the model only conditions on past context — exactly what autoregressive generation requires.

In [ ]:
def softmax(x, axis=-1):
    exp_x = np.exp(x - x.max(axis=axis, keepdims=True))
    return exp_x / exp_x.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, causal=False):
    """Scaled dot-product attention. Q, K, V: (seq_len, d_k)."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if causal:
        seq_len = Q.shape[0]
        mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)
        scores = scores + mask
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

# Toy self-attention on a 4-token sequence (decoder-style causal mask)
seq_len, d_model = 4, 8
X = np.random.randn(seq_len, d_model)

W_q = np.random.randn(d_model, d_model) * 0.1
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1

Q, K, V = X @ W_q, X @ W_k, X @ W_v
output, attn_weights = scaled_dot_product_attention(Q, K, V, causal=True)

tokens = ["The", "cat", "sat", "down"]
print(f"Input shape:  {X.shape}")
print(f"Output shape: {output.shape}")
print(f"\nCausal attention weights (rows=query, cols=key):")
print(np.round(attn_weights, 3))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attn_weights, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(im, label="Attention weight")
ax.set_xticks(range(len(tokens)))
ax.set_xticklabels(tokens)
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens)
ax.set_xlabel("Key (attended to)")
ax.set_ylabel("Query (attending from)")
ax.set_title("Causal self-attention (decoder-only / GPT-style)")
for i in range(len(tokens)):
    for j in range(len(tokens)):
        val = attn_weights[i, j]
        if val > 0.01:
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    color="white" if val > 0.5 else "black", fontsize=9)
plt.tight_layout()
plt.show()

---

## 5. Positional Encoding

Self-attention is **permutation invariant** — reordering tokens changes only row order, not the pairwise relationships computed by dot products. But word order is essential: *"dog bites man"* and *"man bites dog"* share the same tokens in different order.

**Positional encodings** inject sequence position into the representation. They are added to token embeddings before the first attention layer.

### 5.1 Sinusoidal Positional Encoding

The original Transformer uses fixed sinusoidal functions of varying frequency:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

Each dimension oscillates at a different wavelength, giving every position a unique signature. Sinusoidal encodings extrapolate to sequence lengths not seen during training and allow the model to learn relative positions through linear combinations.

### 5.2 Learned Positional Embeddings

GPT-style models use **learned** positional embeddings — a second lookup table indexed by position, trained jointly with token embeddings. This works well when training and inference sequence lengths are similar.

### 5.3 Modern Alternatives

Recent LLMs often use **Rotary Position Embeddings (RoPE)** or **ALiBi** (Attention with Linear Biases), which bake position directly into the attention computation rather than adding to embeddings. RoPE rotates query and key vectors based on position, preserving relative distance information efficiently.

Regardless of method, the goal is the same: **tell the model where each token sits in the sequence** so it can generate coherent, ordered output.

In [ ]:
def sinusoidal_positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    position = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

seq_len, d_model = 50, 64
pe_sinusoidal = sinusoidal_positional_encoding(seq_len, d_model)

# Learned positional embeddings (random init for illustration)
np.random.seed(7)
pe_learned = np.random.randn(seq_len, d_model) * 0.02

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im0 = axes[0].imshow(pe_sinusoidal, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
axes[0].set_xlabel("Embedding dimension")
axes[0].set_ylabel("Position in sequence")
axes[0].set_title("Sinusoidal positional encoding (original Transformer)")
plt.colorbar(im0, ax=axes[0], label="Encoding value")

im1 = axes[1].imshow(pe_learned, aspect="auto", cmap="RdBu")
axes[1].set_xlabel("Embedding dimension")
axes[1].set_ylabel("Position in sequence")
axes[1].set_title("Learned positional embeddings (GPT-style)")
plt.colorbar(im1, ax=axes[1], label="Encoding value")

plt.tight_layout()
plt.show()

# Positional signal added to token embeddings
token_emb = np.random.randn(seq_len, d_model) * 0.1
combined = token_emb + pe_sinusoidal
print(f"Token embeddings:     {token_emb.shape}")
print(f"Positional encoding:  {pe_sinusoidal.shape}")
print(f"Combined input:       {combined.shape}")

---

## 6. Feed Forward Neural Network

After attention mixes information **across** tokens, each position passes through a **position-wise feed-forward network (FFN)** that transforms it **independently**. The same FFN weights apply to every token — only the input vector differs.

$$\text{FFN}(\mathbf{x}) = \text{GELU}(\mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2$$

Typical dimensions: expand to $4 \times d_{\text{model}}$ in the hidden layer, then project back. For $d_{\text{model}} = 768$, the FFN hidden size is 3072.

**Role in generation:** Attention decides *which* tokens to gather information from. The FFN processes and transforms that gathered information — applying nonlinear computation that increases the model's expressive power. Empirically, FFN layers store substantial factual knowledge; some research suggests individual neurons encode specific concepts.

Modern LLMs use **GELU** (Gaussian Error Linear Unit) instead of ReLU for smoother gradients. SwiGLU variants (used in LLaMA, PaLM) replace the single hidden layer with a gated structure for improved quality at similar compute cost.

```
Each token position (in parallel):
  x (d_model) → Linear → GELU → Linear → output (d_model)
                ↑ 4× wider ↑
```

---

## 7. Residual Connections in Transformers

Deep Transformer stacks — GPT-3 has 96 layers, GPT-4 reportedly many more — would be impossible to train without **residual connections** and **layer normalization**.

### 7.1 Residual Connections

Each sublayer (attention or FFN) uses a **skip connection**:

$$\mathbf{x}_{\text{out}} = \mathbf{x}_{\text{in}} + \text{SubLayer}(\mathbf{x}_{\text{in}})$$

The input is added directly to the sublayer output. Gradients can flow through the identity path, avoiding vanishing gradients in very deep networks. The sublayer learns a **residual** — the adjustment to apply — rather than the full transformation.

### 7.2 Layer Normalization

**Layer normalization** normalizes each token's feature vector to zero mean and unit variance:

$$\text{LayerNorm}(\mathbf{x}) = \gamma \odot \frac{\mathbf{x} - \mu}{\sigma + \epsilon} + \beta$$

Unlike batch normalization (which normalizes across the batch), layer norm normalizes across features within each token. This is stable for variable sequence lengths and small batch sizes common in language modeling.

**Pre-LN vs Post-LN:** Original Transformers apply norm after the residual add (Post-LN). GPT-2 and most modern generative models use **Pre-LN** — normalize before each sublayer — which trains more stably at depth.

### 7.3 Full Transformer Block

A single generative Transformer block combines all components:

```
Input: token embeddings + positional encoding
        ↓
  Layer Norm → Multi-Head Causal Self-Attention → Residual Add
        ↓
  Layer Norm → Feed-Forward Network → Residual Add
        ↓
  Output to next block (repeat N times)
        ↓
  Final Layer Norm → Linear projection → vocabulary logits → next token
```

Stacking dozens of these identical blocks, each refining the representation, produces the rich contextual embeddings that drive high-quality text generation.

In [ ]:
def layer_norm(x, eps=1e-5):
    mu = x.mean(axis=-1, keepdims=True)
    sigma = x.std(axis=-1, keepdims=True)
    return (x - mu) / (sigma + eps)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

def multi_head_attention(X, d_model, num_heads, causal=True):
    assert d_model % num_heads == 0
    d_k = d_model // num_heads
    heads = []
    for _ in range(num_heads):
        W_q = np.random.randn(d_model, d_k) * np.sqrt(2.0 / d_model)
        W_k = np.random.randn(d_model, d_k) * np.sqrt(2.0 / d_model)
        W_v = np.random.randn(d_model, d_k) * np.sqrt(2.0 / d_model)
        Q, K, V = X @ W_q, X @ W_k, X @ W_v
        out, _ = scaled_dot_product_attention(Q, K, V, causal=causal)
        heads.append(out)
    concat = np.concatenate(heads, axis=-1)
    W_o = np.random.randn(d_model, d_model) * np.sqrt(2.0 / d_model)
    return concat @ W_o

def feed_forward(x, d_model, d_ff):
    W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)
    W2 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)
    return gelu(x @ W1) @ W2

def transformer_block(x, d_model, num_heads, d_ff):
    """Pre-LN decoder block: norm → sublayer → residual."""
    # Causal self-attention sublayer
    attn_out = multi_head_attention(layer_norm(x), d_model, num_heads, causal=True)
    x = x + attn_out
    # Feed-forward sublayer
    ff_out = feed_forward(layer_norm(x), d_model, d_ff)
    x = x + ff_out
    return x

# Simulate a tiny GPT-style forward pass
d_model, num_heads, d_ff = 32, 4, 128
seq_len = 6
num_layers = 3

token_ids = np.array([101, 4523, 891, 234, 5678, 102])  # pretend vocabulary IDs
vocab_size = 10000
embedding_matrix = np.random.randn(vocab_size, d_model) * 0.02
x = embedding_matrix[token_ids] + sinusoidal_positional_encoding(seq_len, d_model)

print(f"Initial hidden states: {x.shape}")
for layer_idx in range(num_layers):
    x = transformer_block(x, d_model, num_heads, d_ff)
    print(f"After block {layer_idx + 1}: mean={x.mean():.4f}, std={x.std():.4f}")

# Project final hidden state at last position to vocabulary logits (next-token prediction)
W_lm = embedding_matrix.T  # tied embeddings
logits = layer_norm(x)[-1] @ W_lm
probs = softmax(logits)
top5 = np.argsort(probs)[-5:][::-1]
print(f"\nNext-token logits shape: {logits.shape}")
print(f"Top-5 predicted token IDs: {top5}")
print(f"Top-5 probabilities:     {np.round(probs[top5], 4)}")

---

## 8. Summary

The **Transformer** architecture enabled modern generative AI by replacing sequential recurrence with parallel **self-attention**. Decoder-only stacks trained on next-token prediction power today's LLMs; encoder-decoder variants handle conditional generation tasks like translation and summarization; cross-attention connects text encoders to image diffusion models.

Each generative Transformer block combines:

- **Input embeddings** — discrete tokens mapped to continuous vectors
- **Positional encoding** — sinusoidal, learned, or rotary signals that preserve order
- **Multi-head causal self-attention** — scaled dot-product attention with Q/K/V projections, masked so each token sees only past context
- **Feed-forward networks** — position-wise nonlinear transforms applied after attention
- **Residual connections and layer normalization** — stable gradient flow through deep stacks

Understanding these components clarifies how GPT-style models process a prompt, build contextual representations layer by layer, and produce the next-token distribution that drives autoregressive generation. Later notebooks in this section explore tokens, context windows, model families, and inference parameters that build directly on this architecture.